# Notebook 51 — LLM Adapter Layer

Demonstrates `multigen.llm_adapter`: mock adapters, caching, token budgets,
context-window management, structured output parsing, and the router facade.
No real API keys are required — all calls go through `EchoAdapter`.

In [ ]:
import sys, importlib.util
sys.path.insert(0, '../sdk')

def load(name):
    spec = importlib.util.spec_from_file_location(
        f'multigen.{name}', f'../sdk/multigen/{name}.py')
    mod = importlib.util.module_from_spec(spec)
    sys.modules[f'multigen.{name}'] = mod
    spec.loader.exec_module(mod)
    return mod

llm = load('llm_adapter')
print('llm_adapter loaded OK')

## 1. EchoAdapter — basic completion

In [ ]:
import asyncio

adapter = llm.EchoAdapter(response_template="Echo: {last_message}")

req = llm.LLMRequest(
    messages=[llm.LLMMessage(role='user', content='What is the capital of France?')],
    model='echo-1',
)

response = asyncio.run(adapter.complete(req))

print('content      :', response.content)
print('input_tokens :', response.input_tokens)
print('output_tokens:', response.output_tokens)
print('total_tokens :', response.total_tokens)
print('latency_ms   :', round(response.latency_ms, 3), 'ms')

## 2. LLMMessage + LLMRequest — 3-turn conversation and cache_key

In [ ]:
msgs = [
    llm.LLMMessage(role='system',    content='You are a helpful assistant.'),
    llm.LLMMessage(role='user',      content='Tell me about the Roman Empire.'),
    llm.LLMMessage(role='assistant', content='The Roman Empire was founded in 27 BC.'),
]

request_a = llm.LLMRequest(messages=msgs, model='echo-1', temperature=0.5)
request_b = llm.LLMRequest(messages=msgs, model='echo-1', temperature=0.5)  # identical
request_c = llm.LLMRequest(messages=msgs, model='echo-1', temperature=0.9)  # different temp

print('cache_key A:', request_a.cache_key())
print('cache_key B:', request_b.cache_key(), '  <- same as A?', request_a.cache_key() == request_b.cache_key())
print('cache_key C:', request_c.cache_key(), '  <- different from A?', request_a.cache_key() != request_c.cache_key())
print('variables extracted from message content:', [m.to_dict() for m in msgs])

## 3. LLMCache — hit rate and cached flag

In [ ]:
cache = llm.LLMCache(max_size=100)

req1 = llm.LLMRequest(
    messages=[llm.LLMMessage(role='user', content='Hello world')],
    model='echo-1',
)
req2 = llm.LLMRequest(
    messages=[llm.LLMMessage(role='user', content='Goodbye world')],
    model='echo-1',
)

adapter = llm.EchoAdapter()

# First call for req1 — miss
r1 = asyncio.run(adapter.complete(req1))
cache.set(req1, r1)

# First call for req2 — miss
r2 = asyncio.run(adapter.complete(req2))
cache.set(req2, r2)

print('Stats after 2 misses:', cache.stats())

# Second call for req1 — hit
hit = cache.get(req1)
print('Cached result:', hit.content)
print('cached flag  :', hit.cached)

# Another hit
cache.get(req1)

print('Stats after 2 hits  :', cache.stats())
print('hit_rate:', round(cache.hit_rate, 2))

## 4. TokenBudget — consume tokens and trigger BudgetExceededError

In [ ]:
budget = llm.TokenBudget(max_tokens=200)

budget.consume(100)
print('After consuming 100 — used:', budget.used, '| remaining:', budget.remaining)

budget.consume(50)
print('After consuming  50 — used:', budget.used, '| remaining:', budget.remaining)

# Trigger BudgetExceededError
try:
    budget.consume(60)   # 100+50+60 = 210 > 200
except llm.BudgetExceededError as exc:
    print('BudgetExceededError raised:', exc)

## 5. ContextWindowManager — truncate oldest messages to fit limit

In [ ]:
# 10 messages, each ~40 chars.  At 0.25 tokens/char that is ~10 tokens each.
# Set max_tokens=30 so only ~3 non-system messages can fit.
mgr = llm.ContextWindowManager(max_tokens=30, strategy='truncate_oldest')

messages = [
    llm.LLMMessage(role='system', content='You are a helpful assistant.')
] + [
    llm.LLMMessage(role='user' if i % 2 == 0 else 'assistant',
                   content=f'Turn {i}: some content here.')
    for i in range(9)
]

print(f'Original message count : {len(messages)}')
print(f'Estimated tokens (full): {mgr.estimate_tokens(messages)}')

fitted = mgr.fit(messages)
print(f'Fitted  message count  : {len(fitted)}')
print('Kept messages:')
for m in fitted:
    print(f'  [{m.role:9s}] {m.content[:50]}')

## 6. LLMRouter — wire cache, complete through router, verify cache hit on second call

In [ ]:
cache = llm.LLMCache()
router = llm.LLMRouter(cache=cache)
router.add_adapter('echo', llm.EchoAdapter())

print('Registered adapters:', router.adapters())

msgs = [{'role': 'user', 'content': 'What is 2 + 2?'}]

# First call — cache miss
r1 = asyncio.run(router.complete(msgs, adapter='echo', temperature=0.0))
print('\n--- First call (cache miss) ---')
print('content:', r1.content)
print('cached :', r1.cached)
print('cache stats:', cache.stats())

# Second identical call — cache hit
r2 = asyncio.run(router.complete(msgs, adapter='echo', temperature=0.0))
print('\n--- Second call (cache hit) ---')
print('content:', r2.content)
print('cached :', r2.cached)
print('cache stats:', cache.stats())

## 7. StructuredOutputParser — parse valid JSON and raise ValueError on missing required field

In [ ]:
schema = {'type': 'object', 'required': ['name', 'score']}
parser = llm.StructuredOutputParser(schema=schema)

# Valid JSON
valid_json = '{"name": "Alice", "score": 0.95, "notes": "top performer"}'
parsed = parser.parse(valid_json)
print('Parsed dict:', parsed)
print('Name :', parsed['name'])
print('Score:', parsed['score'])

# JSON inside markdown fence
fenced = '```json\n{"name": "Bob", "score": 0.82}\n```'
parsed_fenced = parser.parse(fenced)
print('\nParsed from fenced markdown:', parsed_fenced)

# Missing required field raises ValueError
missing_field_json = '{"name": "Charlie"}'
try:
    parser.parse(missing_field_json)
except ValueError as exc:
    print('\nValueError (missing field):', exc)